📂 KAIST Multispectral Pedestrian Dataset Visualization

### Step 1: Import Libraries

In [ ]:
import cv2
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import os

from tqdm import tqdm

# Define color map for each class (BGR format)
CLASS_COLORS = {
    'person': (0, 255, 0),      # Green
    'cyclist': (255, 255, 0),   # Cyan
    'person?': (0, 165, 255),   # Orange (Unclear cases)
    'people': (0, 0, 255),      # Red (Group of people)
    'ignore': (128, 128, 128)   # Gray
}
DEFAULT_COLOR = (255, 255, 255) # White (for any other classes)

### Step 2: Define XML Parser
The KAIST dataset provides bounding boxes in `[x, y, w, h]` format. This function converts them to `[x1, y1, x2, y2]` for easier visualization with OpenCV.

In [ ]:
def parse_kaist_xml(xml_path):
    """
    Parses the KAIST XML annotation format.
    Args:
        xml_path (str): Path to the XML file.
    Returns:
        list: A list of dictionaries containing 'name' and 'bbox' [x1, y1, x2, y2].
    """
    if not os.path.exists(xml_path):
        print(f"Error: XML file not found at {xml_path}")
        return []

    tree = ET.parse(xml_path)
    root = tree.getroot()

    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        bndbox = obj.find('bndbox')

        # KAIST dataset format: x, y, width, height
        x = int(bndbox.find('x').text)
        y = int(bndbox.find('y').text)
        w = int(bndbox.find('w').text)
        h = int(bndbox.find('h').text)

        objects.append({
            'name': name,
            'bbox': [x, y, x + w, y + h] # Convert to [x_min, y_min, x_max, y_max]
        })
    return objects

### Step 3: Define Visualization Function
This function displays the Visible (RGB) and Thermal (LWIR) images side-by-side with overlaid bounding boxes.

In [ ]:
def visualize_kaist_pair(vis_path, lw_path, xml_path):
    """
    Visualizes multispectral image pairs with annotations.
    """
    # Load images
    vis_img = cv2.imread(vis_path)
    lw_img = cv2.imread(lw_path)

    if vis_img is None or lw_img is None:
        print("Error: Could not load images. Please check the file paths.")
        return

    # Convert BGR (OpenCV default) to RGB for Matplotlib
    vis_img = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)
    lw_img = cv2.cvtColor(lw_img, cv2.COLOR_BGR2RGB)

    # Parse annotations from XML
    boxes = parse_kaist_xml(xml_path)

    # Set up the figure for side-by-side display
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    display_data = [
        (vis_img, "Visible (RGB)"),
        (lw_img, "Thermal (LWIR)")
    ]

    for i, (img, title) in enumerate(display_data):
        for obj in boxes:
            name = obj['name']
            box = obj['bbox']

            # Get color for the specific class, default to white if not found
            color = CLASS_COLORS.get(name.lower(), DEFAULT_COLOR)

            # Draw Bounding Box
            cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), color, 2)

            # Draw Label with specific color
            cv2.putText(img, name, (box[0], box[1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        axes[i].imshow(img)
        axes[i].set_title(title, fontsize=15)
        axes[i].axis('off') # Hide axes for cleaner look

    plt.tight_layout()
    plt.show()

### Step 4: Execution Example
Update the paths below to match your local directory structure.

In [ ]:
# --- User Configuration ---
# Example paths (Update these according to your dataset directory)
xml_sample = "annotations-xml-new-sanitized/set04/V001/I00097.xml"
vis_sample = "images/set04/V001/visible/I00097.jpg"
lw_sample  = "images/set04/V001/lwir/I00097.jpg"

# Run the visualization
visualize_kaist_pair(vis_sample, lw_sample, xml_sample)

### Step 5: Export Sequence to Video using List File
This function reads a text file containing the list of frames and compiles them into a side-by-side video.

In [ ]:
def export_video_from_list(list_file_path, base_image_dir, base_xml_dir, output_path, fps=20):
    """
    Creates a video from a specific list of frames provided in a text file.
    Args:
        list_file_path (str): Path to "train-preview-01.txt"
        base_image_dir (str): Root directory for images (contains setXX folders)
        base_xml_dir (str): Root directory for annotations
        output_path (str): Output video path (e.g., 'preview.mp4')
        fps (int): Frames per second
    """
    # 1. Read the frame list from the text file
    if not os.path.exists(list_file_path):
        print(f"Error: List file not found at {list_file_path}")
        return

    with open(list_file_path, 'r') as f:
        # Read lines and strip whitespace/newlines
        frame_list = [line.strip().split('/') for line in f.readlines() if line.strip()]

    if not frame_list:
        print("The list file is empty.")
        return

    # 2. Initialize Video Writer
    # Side-by-Side: Width (640*2 = 1280), Height (512)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (1280, 512))

    print(f"Processing {len(frame_list)} frames...")

    for sid, vid, iid in tqdm(frame_list, desc="Generating Video", unit="frame"):
        # frame_rel_path looks like: "set04/V001/I00000"

        # Construct full paths
        vis_path = os.path.join(base_image_dir, sid, vid, "visible", iid + ".jpg")
        lw_path = os.path.join(base_image_dir, sid, vid, "lwir", iid + ".jpg")

        # Construct XML path
        xml_path = os.path.join(base_xml_dir, sid, vid, iid + ".xml")

        # 3. Load Images
        img_v = cv2.imread(vis_path)
        img_l = cv2.imread(lw_path)

        if img_v is None or img_l is None:
            print(f"Warning: Image not found for {vis_path} or {lw_path}. Skipping.")
            continue

        # 4. Parse and Draw Annotations
        boxes = parse_kaist_xml(xml_path)
        for obj in boxes:
            name = obj['name'].lower()
            box = obj['bbox']

            # --- Multi-class coloring logic ---
            color = CLASS_COLORS.get(name, DEFAULT_COLOR)

            for img in [img_v, img_l]:
                cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), color, 2)
                cv2.putText(img, name, (box[0], box[1] - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # 5. Concatenate and Write to Video
        combined_frame = cv2.hconcat([img_v, img_l])
        out.write(combined_frame)

    out.release()
    print(f"Video saved successfully at: {output_path}")

In [ ]:
# --- Execution Example ---
list_txt = "imageSets/train-preview-01.txt"
image_root = "images"
xml_root = "annotations-xml-new-sanitized"
output_name = "preview_visualization.mp4"

export_video_from_list(list_txt, image_root, xml_root, output_name)